# SelvaSonic ML — Comparativa Final de los 3 Modelos

**Autores:** Laura Ruiz Arango · Jose Aldair Molina Méndez  
**Asignatura:** Aprendizaje Automático  
**Profesor:** Alcides Montoya  
**Fecha:** Junio 2026  

---

## Propósito

Este notebook es la **vista ejecutiva final** del proyecto SelvaSonic: compara lado a lado
los tres modelos entrenados sobre el mismo dataset de vocalizaciones amazónicas.

| Modelo | Val acc | Test acc | Características |
|---|---|---|---|
| `SelvaSonicCNN` (baseline) | 0.7419 | 0.6322 | CNN base, sin balance |
| `SelvaSonicCNNAttention` v1 | 0.7829 | 0.6873 | + Multi-Head Self-Attention |
| `SelvaSonicCNNAttention` v2 | 0.7653 | **0.7036** | + Class weights + Label smoothing α=0.1 |

La pregunta central: **¿qué aportó cada técnica y a qué costo?**
Respondemos con evidencia cuantitativa: métricas globales, F1 por clase, matrices de confusión,
correlación datos-rendimiento, calibración de probabilidades y curvas PR para clases raras.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path
from typing import TypedDict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import pearsonr
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)
from torch.utils.data import DataLoader

# Asegurar raíz del proyecto en sys.path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, NUM_CLASSES
from src.dataset import create_dataloaders
from src.model import SelvaSonicCNN, SelvaSonicCNNAttention

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Rutas a checkpoints ──────────────────────────────────────────────────
CKPT_BASELINE = PROJECT_ROOT / 'results/runs/baseline_S3_v2_20260527_0118/best.pth'
CKPT_V1       = PROJECT_ROOT / 'results/runs/attention_S4_v1_20260601_0334/best.pth'
CKPT_V2       = PROJECT_ROOT / 'results/runs/attention_S4_v2_20260602_1332/best.pth'
OUT_DIR       = PROJECT_ROOT / 'results/comparativa'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for nombre, path in [('baseline', CKPT_BASELINE), ('attention_v1', CKPT_V1), ('attention_v2', CKPT_V2)]:
    if not path.exists():
        raise FileNotFoundError(f'Checkpoint faltante para {nombre}: {path}')
    print(f'  {nombre}: {path.parent.name} — OK')

# ── Paleta de colores consistente con notebooks anteriores ───────────────
COLOR_BASELINE = '#FD79A8'   # rosa
COLOR_V1       = '#6C5CE7'   # púrpura
COLOR_V2       = '#00B894'   # verde
COLORES        = [COLOR_BASELINE, COLOR_V1, COLOR_V2]
NOMBRES        = ['Baseline', 'Attention v1', 'Attention v2 (balanced)']
CLAVES         = ['baseline', 'attention_v1', 'attention_v2']

# Archivos originales por clase (fuente: dataset de Xeno-Canto + ESC-50)
ARCHIVOS_POR_CLASE: dict[str, int] = {
    'no_ave': 1600,
    'Celeus_grammicus': 28,
    'Chordeiles_pusillus': 21,
    'Crypturellus_cinereus': 48,
    'Crypturellus_undulatus': 29,
    'Frederickena_fulva': 20,
    'Glaucidium_brasilianum': 22,
    'Lipaugus_vociferans': 36,
    'Ramphastos_tucanus': 31,
    'Rupornis_magnirostris': 20,
    'Trogon_viridis': 79,
}

# Clases raras para análisis focalizados (≤28 archivos de entrenamiento)
CLASES_RARAS = [
    'Frederickena_fulva',
    'Rupornis_magnirostris',
    'Chordeiles_pusillus',
    'Glaucidium_brasilianum',
    'Celeus_grammicus',
]

## Setup: carga del test DataLoader

Usamos la misma semilla (`random_state=42`) y el mismo `DATA_DIR` que todos los notebooks
anteriores. Esto garantiza que el test set es **idéntico** al usado durante el entrenamiento,
y que las métricas son directamente comparables.

In [ ]:
_, _, test_loader, label_map = create_dataloaders(
    DATA_DIR / 'raw',
    batch_size=32,
    num_workers=0,
    random_state=42,
    verbose=False,
)

# Orden canónico de clases: índice 0..10
CLASES: list[str] = [nombre for nombre, _ in sorted(label_map.items(), key=lambda x: x[1])]
IDX_A_CLASE: dict[int, str] = {v: k for k, v in label_map.items()}

n_test = len(test_loader.dataset)
print(f'Test set: {n_test} clips | {len(test_loader)} batches | {len(CLASES)} clases')
print(f'Clases: {CLASES}\n')

# Distribución de clips por clase en test
dist = Counter(int(label) for _, label in test_loader.dataset)
print('Distribución test set:')
for idx in range(NUM_CLASSES):
    print(f'  {idx:2d}  {CLASES[idx]:<30}  {dist[idx]:4d} clips')

## Función helper: `evaluar_modelo`

Una única función reutilizable para los 3 modelos. Devuelve un diccionario tipado con
todo lo necesario para las secciones de análisis.

In [ ]:
class ResultadoModelo(TypedDict):
    nombre: str
    y_true: np.ndarray
    y_pred: np.ndarray
    y_proba: np.ndarray           # (N, NUM_CLASSES) — softmax
    accuracy: float
    macro_f1: float
    macro_precision: float
    macro_recall: float
    f1_per_class: np.ndarray      # (NUM_CLASSES,)
    precision_per_class: np.ndarray
    recall_per_class: np.ndarray
    confusion_matrix: np.ndarray  # (NUM_CLASSES, NUM_CLASSES) normalizada por fila


def evaluar_modelo(
    *,
    modelo: nn.Module,
    test_loader: DataLoader,
    device: torch.device,
    nombre: str,
) -> ResultadoModelo:
    """Evalúa un modelo en el test set y devuelve todas las métricas necesarias."""
    modelo.eval()
    acum_true, acum_pred, acum_proba = [], [], []

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            logits = modelo(x_batch.to(device))
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            acum_proba.append(probs)
            acum_pred.extend(probs.argmax(axis=1))
            acum_true.extend(y_batch.numpy())

    y_true  = np.array(acum_true, dtype=int)
    y_pred  = np.array(acum_pred, dtype=int)
    y_proba = np.vstack(acum_proba)           # (N, NUM_CLASSES)

    report = classification_report(
        y_true, y_pred,
        target_names=CLASES,
        output_dict=True,
        zero_division=0,
    )
    cm_raw  = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True).clip(min=1)

    return {
        'nombre': nombre,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'accuracy': float((y_true == y_pred).mean()),
        'macro_f1': float(report['macro avg']['f1-score']),
        'macro_precision': float(report['macro avg']['precision']),
        'macro_recall': float(report['macro avg']['recall']),
        'f1_per_class':        np.array([report[c]['f1-score']  for c in CLASES]),
        'precision_per_class': np.array([report[c]['precision'] for c in CLASES]),
        'recall_per_class':    np.array([report[c]['recall']    for c in CLASES]),
        'confusion_matrix': cm_norm,
    }

## Cargar checkpoints y evaluar en test set

Instanciamos la arquitectura correcta para cada checkpoint:
- `SelvaSonicCNN` para el baseline (28 keys en `model_state_dict`)
- `SelvaSonicCNNAttention` para v1 y v2 (35 keys, incluye `pos_encoding` y capas de atención)

In [ ]:
def _cargar_modelo(
    *,
    arquitectura: type,
    checkpoint_path: Path,
    device: torch.device,
) -> nn.Module:
    """Instancia la arquitectura y carga los pesos del checkpoint."""
    ckpt   = torch.load(str(checkpoint_path), map_location=device, weights_only=False)
    modelo = arquitectura(num_classes=NUM_CLASSES)
    modelo.load_state_dict(ckpt['model_state_dict'])
    modelo.to(device).eval()
    acc_str = f"{ckpt.get('best_val_acc', 0):.4f}" if 'best_val_acc' in ckpt else '?'
    print(f"  {checkpoint_path.parent.name} | epoch={ckpt.get('epoch','?')} | best_val_acc={acc_str}")
    return modelo


print('Cargando checkpoints...')
modelo_baseline = _cargar_modelo(arquitectura=SelvaSonicCNN,           checkpoint_path=CKPT_BASELINE, device=DEVICE)
modelo_v1       = _cargar_modelo(arquitectura=SelvaSonicCNNAttention,   checkpoint_path=CKPT_V1,       device=DEVICE)
modelo_v2       = _cargar_modelo(arquitectura=SelvaSonicCNNAttention,   checkpoint_path=CKPT_V2,       device=DEVICE)

print('\nEvaluando en test set...')
resultados: dict[str, ResultadoModelo] = {}
for clave, modelo in zip(CLAVES, [modelo_baseline, modelo_v1, modelo_v2]):
    print(f'  {clave}...', end=' ', flush=True)
    resultados[clave] = evaluar_modelo(
        modelo=modelo,
        test_loader=test_loader,
        device=DEVICE,
        nombre=clave,
    )
    r = resultados[clave]
    print(f'acc={r["accuracy"]:.4f}  macro_f1={r["macro_f1"]:.4f}')

print('\nEvaluación completa.')

---
## Sección 1 — Tabla de métricas globales

Ante un dataset tan desbalanceado (`no_ave` = 1 600 archivos vs. Rupornis = 20),
el **accuracy** favorece artificialmente a quien aprende bien la clase mayoritaria.
El **macro F1** pondera igual a todas las clases, independientemente de su soporte,
y por eso es la métrica de referencia para medir si el balance de clases funcionó.

Esperamos que v2 lidere en **test accuracy** y en **macro F1**; si macro F1 crece más
que accuracy, es evidencia directa de que las clases raras mejoraron.

In [ ]:
# ── Tabla de métricas globales ───────────────────────────────────────────
metricas_globales = {
    clave: {
        'accuracy':         resultados[clave]['accuracy'],
        'macro_f1':         resultados[clave]['macro_f1'],
        'macro_precision':  resultados[clave]['macro_precision'],
        'macro_recall':     resultados[clave]['macro_recall'],
    }
    for clave in CLAVES
}

df_global = pd.DataFrame(metricas_globales, index=['accuracy', 'macro_f1', 'macro_precision', 'macro_recall']).T
df_global.index = NOMBRES

# Resaltar la mejor celda de cada columna
def resaltar_max(col):
    return ['font-weight: bold; background-color: #d4f5d4' if v == col.max() else '' for v in col]

display(df_global.style.apply(resaltar_max).format('{:.4f}'))

df_global.to_csv(OUT_DIR / 'tabla_metricas_globales.csv')
print(f'Guardado: {OUT_DIR / "tabla_metricas_globales.csv"}')

### Interpretación

Analiza la tabla generada arriba y responde:
- ¿Qué modelo lidera en accuracy? ¿y en macro F1?
- ¿La diferencia entre accuracy y macro F1 es coherente con el desbalance del dataset?
- Si v2 gana en ambas métricas: los class weights mejoraron las clases raras SIN sacrificar las frecuentes.
- Si v2 gana en macro F1 pero pierde en accuracy: hubo un trade-off explícito (exactamente lo que class weights intentan producir).

---
## Sección 2 — F1 por clase

La hipótesis central de v2 es que los **class weights** aumentan el F1 de las clases
con menos datos de entrenamiento. Las clases candidatas a mejorar son las que tienen
20-28 archivos: Frederickena, Rupornis, Chordeiles, Glaucidium, Celeus.

Las clases se ordenan de **menor a mayor** número de archivos de entrenamiento.
Si el peso funciona, deberían verse saltos más pronunciados en el extremo izquierdo del gráfico.

In [ ]:
# Ordenar clases de menos a más archivos (excluye no_ave para claridad visual)
orden_clases = sorted(
    [c for c in CLASES if c != 'no_ave'],
    key=lambda c: ARCHIVOS_POR_CLASE.get(c, 0),
) + ['no_ave']
idx_orden = [CLASES.index(c) for c in orden_clases]

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('#FAFAFA')

n_clases = len(orden_clases)
x = np.arange(n_clases)
ancho = 0.26
offsets = [-ancho, 0, ancho]

for i, (clave, color, nombre) in enumerate(zip(CLAVES, COLORES, NOMBRES)):
    f1_vals = resultados[clave]['f1_per_class'][idx_orden]
    bars = ax.bar(x + offsets[i], f1_vals, ancho, label=nombre,
                  color=color, alpha=0.85, edgecolor='white')
    # Anotar valor encima de cada barra
    for bar, val in zip(bars, f1_vals):
        if val > 0.02:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=5.5,
            )

ax.set_xticks(x)
ax.set_xticklabels(
    [f"{c}\n({ARCHIVOS_POR_CLASE.get(c,'?')} arch.)" for c in orden_clases],
    fontsize=7.5, rotation=30, ha='right',
)
ax.set_ylabel('F1-score', fontsize=11)
ax.set_title(
    'F1 por clase — Baseline vs Attention v1 vs v2 (balanced)\n'
    'Clases ordenadas de menor a mayor cantidad de archivos de entrenamiento',
    fontsize=11,
)
ax.set_ylim(0, 1.12)
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.savefig(OUT_DIR / 'f1_por_clase.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Guardado: {OUT_DIR / "f1_por_clase.png"}')

# Tabla con deltas
df_f1_clase = pd.DataFrame({
    'clase': orden_clases,
    'archivos': [ARCHIVOS_POR_CLASE.get(c, 0) for c in orden_clases],
    'f1_baseline':    resultados['baseline']['f1_per_class'][idx_orden],
    'f1_attention_v1': resultados['attention_v1']['f1_per_class'][idx_orden],
    'f1_attention_v2': resultados['attention_v2']['f1_per_class'][idx_orden],
})
df_f1_clase['delta_v2_vs_v1'] = df_f1_clase['f1_attention_v2'] - df_f1_clase['f1_attention_v1']
df_f1_clase['delta_v2_vs_base'] = df_f1_clase['f1_attention_v2'] - df_f1_clase['f1_baseline']
print(df_f1_clase.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

### Interpretación

- **Clases raras (≤28 archivos):** observa si las barras verdes (v2) son más altas que las púrpuras (v1) en el extremo izquierdo.
  Eso confirma que los class weights compensaron el desbalance.
- **Clases frecuentes:** si v2 baja en `no_ave` o `Trogon_viridis`, es el trade-off esperado — el modelo
  sacrifica algo de precisión en la clase fácil para mejorar en las difíciles.
- **Delta v2 vs v1:** compara la columna `delta_v2_vs_v1` en la tabla. Un delta positivo en clases raras
  es la evidencia cuantitativa de que los class weights funcionaron.

---
## Sección 3 — Matrices de confusión comparativas

Matrices normalizadas **por fila** (cada celda = recall de esa clase hacia otra).
La diagonal es el recall por clase. Una diagonal más uniforme y más intensa en v2
indica que el balance de clases mejoró el recall de las clases que antes eran ignoradas.

Patrones esperados:
- Baseline: diagonal débil en clases raras (Rupornis, Frederickena).
- v1: diagonal más fuerte globalmente, gracias al attention.
- v2: diagonal más uniforme entre todas las clases, gracias a los class weights.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#FAFAFA')

titulos = ['Baseline', 'Attention v1', 'Attention v2 (balanced)']

for ax, clave, titulo in zip(axes, CLAVES, titulos):
    cm = resultados[clave]['confusion_matrix']
    sns.heatmap(
        cm, ax=ax,
        annot=True, fmt='.2f',
        cmap='Blues',
        vmin=0, vmax=1,
        xticklabels=CLASES,
        yticklabels=CLASES,
        linewidths=0.3,
        linecolor='white',
        cbar=(clave == 'attention_v2'),
        annot_kws={'size': 6},
    )
    ax.set_title(titulo, fontsize=12, pad=10)
    ax.set_xlabel('Predicho', fontsize=9)
    ax.set_ylabel('Real', fontsize=9)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)

plt.suptitle(
    'Matrices de confusión normalizadas por fila (recall por clase)',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.savefig(OUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Guardado: {OUT_DIR / "confusion_matrices.png"}')

# Recall diagonal (precisión por clase) para comparar cuantitativamente
print('\nRecall diagonal por clase:')
print(f'{"Clase":<30}  {"Baseline":>10}  {"Att. v1":>10}  {"Att. v2":>10}')
for i, clase in enumerate(CLASES):
    r_b  = resultados['baseline']['confusion_matrix'][i, i]
    r_v1 = resultados['attention_v1']['confusion_matrix'][i, i]
    r_v2 = resultados['attention_v2']['confusion_matrix'][i, i]
    print(f'{clase:<30}  {r_b:>10.3f}  {r_v1:>10.3f}  {r_v2:>10.3f}')

### Interpretación

- Busca las filas correspondientes a las clases raras: ¿la diagonal es más intensa en v2?
- Observa el patrón de errores frecuentes: ¿hay pares de clases que se confunden entre sí?
  Los dos Crypturellus (C. cinereus y C. undulatus) son candidatos clásicos.
- Una clase con recall cercano a 0 en baseline pero >0.3 en v2 es la evidencia más convincente
  de que los class weights rescataron una clase del olvido.

---
## Sección 4 — F1 vs cantidad de datos de entrenamiento

En el notebook 04 se encontró una **correlación de Pearson alta** entre número de archivos
por clase y su F1: más datos → mejor F1. Eso era la huella del desbalance.

Si los **class weights** de v2 funcionaron, la correlación debería **debilitarse**:
el modelo ya no depende tanto del volumen de datos porque la loss compensa la desigualdad.
Un r más bajo en v2 que en baseline o v1 confirma este efecto.

In [ ]:
# Excluir no_ave del scatter: tiene 1600 archivos y distorsiona la escala
clases_aves = [c for c in CLASES if c != 'no_ave']
idx_aves = [CLASES.index(c) for c in clases_aves]
n_arch = np.array([ARCHIVOS_POR_CLASE[c] for c in clases_aves])

fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor('#FAFAFA')

corrs: dict[str, float] = {}
for clave, color, nombre in zip(CLAVES, COLORES, NOMBRES):
    f1_aves = resultados[clave]['f1_per_class'][idx_aves]
    r, _ = pearsonr(n_arch, f1_aves)
    corrs[clave] = r
    ax.scatter(
        n_arch, f1_aves,
        s=120, c=color, alpha=0.8,
        edgecolors='#2D3436', linewidths=0.6,
        label=f'{nombre}  (r = {r:.3f})',
        zorder=3,
    )

# Líneas que conectan los 3 modelos para la misma clase
f1_matrix = np.vstack([
    resultados[clave]['f1_per_class'][idx_aves] for clave in CLAVES
])
for j in range(len(clases_aves)):
    ax.plot(
        [n_arch[j]] * 3, f1_matrix[:, j],
        '-', color='gray', alpha=0.25, linewidth=1, zorder=2,
    )

# Anotar clase al lado del punto v2
f1_v2 = resultados['attention_v2']['f1_per_class'][idx_aves]
for j, clase in enumerate(clases_aves):
    ax.annotate(
        clase.replace('_', ' ')[:18],
        (n_arch[j], f1_v2[j]),
        fontsize=7, alpha=0.75,
        xytext=(6, 3), textcoords='offset points',
    )

ax.set_xlabel('Archivos de entrenamiento (escala log)', fontsize=11)
ax.set_ylabel('F1-score en test', fontsize=11)
ax.set_xscale('log')
ax.set_title(
    'F1 vs cantidad de datos\n'
    'Si r baja en v2 → class weights redujeron la dependencia del volumen de datos',
    fontsize=11,
)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'f1_vs_datos.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Guardado: {OUT_DIR / "f1_vs_datos.png"}')

print('\nCorrelación Pearson F1 vs archivos (sin no_ave):')
for clave, nombre in zip(CLAVES, NOMBRES):
    print(f'  {nombre:<30}  r = {corrs[clave]:.3f}')

### Interpretación

- **r alto** (>0.7): el rendimiento sigue determinado por el volumen de datos → el desbalance persiste.
- **r bajo** (<0.5 en v2 vs. >0.7 en baseline): los class weights rompieron la dependencia.
- Las líneas grises verticales muestran cuánto «subió» o «bajó» cada clase entre modelos.
  Clases que suben claramente en v2 son las que más se beneficiaron del balance.

---
## Sección 5 — Calibración de probabilidades

En el notebook 11 detectamos **overconfidence** en v1: el modelo asignaba probabilidades
muy cercanas a 1.0 incluso en audios ambiguos. El **label smoothing** (α=0.1) de v2
está diseñado específicamente para corregir esto: penaliza que el modelo se acerque
demasiado a certeza total.

Medimos calibración con el **Expected Calibration Error (ECE)**:
$$ECE = \sum_{m=1}^{M} \frac{|B_m|}{N} \cdot |\text{acc}(B_m) - \text{conf}(B_m)|$$

Un ECE bajo significa que cuando el modelo dice «tengo 80% de confianza», realmente
acierta ~80% del tiempo. Un modelo perfectamente calibrado tendría ECE = 0.

**Esperamos:** ECE(v2) < ECE(v1) < ECE(baseline), y distribución de confianza
más spread en v2 (menos masa acumulada cerca de 1.0).

In [ ]:
def compute_ece(
    *,
    y_true: np.ndarray,
    y_proba: np.ndarray,
    n_bins: int = 15,
) -> float:
    """Calcula el Expected Calibration Error con bins uniformes en [0, 1]."""
    max_probs  = y_proba.max(axis=1)
    is_correct = (y_proba.argmax(axis=1) == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece  = 0.0
    n    = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (max_probs >= lo) & (max_probs < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(is_correct[mask].mean() - max_probs[mask].mean())
    return float(ece)


N_BINS = 15
eces: dict[str, float] = {}
for clave in CLAVES:
    eces[clave] = compute_ece(
        y_true=resultados[clave]['y_true'],
        y_proba=resultados[clave]['y_proba'],
        n_bins=N_BINS,
    )

print('ECE por modelo (menor = mejor calibrado):')
for clave, nombre in zip(CLAVES, NOMBRES):
    print(f'  {nombre:<35}  ECE = {eces[clave]:.4f}')

# ── Reliability diagrams ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#FAFAFA')
bins_edges = np.linspace(0, 1, N_BINS + 1)
bin_centers = (bins_edges[:-1] + bins_edges[1:]) / 2

for ax, clave, color, nombre in zip(axes, CLAVES, COLORES, NOMBRES):
    y_true  = resultados[clave]['y_true']
    y_proba = resultados[clave]['y_proba']
    max_probs  = y_proba.max(axis=1)
    is_correct = (y_proba.argmax(axis=1) == y_true).astype(float)

    acc_bins, counts = [], []
    for lo, hi in zip(bins_edges[:-1], bins_edges[1:]):
        mask = (max_probs >= lo) & (max_probs < hi)
        acc_bins.append(is_correct[mask].mean() if mask.sum() > 0 else np.nan)
        counts.append(mask.sum())
    acc_bins = np.array(acc_bins)

    ax2 = ax.twinx()
    ax2.bar(bin_centers, counts, width=1/N_BINS, color=color, alpha=0.18, edgecolor='none')
    ax2.set_ylabel('Nº muestras', fontsize=8, color='gray')
    ax2.tick_params(axis='y', labelcolor='gray', labelsize=7)

    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.6, label='Perfectamente calibrado')
    ax.plot(bin_centers, acc_bins, 'o-', color=color, lw=2, ms=5, label='Modelo')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Confianza (max softmax)', fontsize=9)
    ax.set_ylabel('Precisión real', fontsize=9)
    ax.set_title(f'{nombre}\nECE = {eces[clave]:.4f}', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Reliability diagrams — calibración de probabilidades', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'calibracion.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Guardado: {OUT_DIR / "calibracion.png"}')

# ── Distribución de confianza (max softmax) ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#FAFAFA')
for clave, color, nombre in zip(CLAVES, COLORES, NOMBRES):
    max_probs = resultados[clave]['y_proba'].max(axis=1)
    ax.hist(max_probs, bins=40, range=(0, 1), histtype='stepfilled',
            color=color, alpha=0.45, edgecolor=color, label=nombre, density=True)
ax.set_xlabel('Confianza máxima (max softmax probability)', fontsize=11)
ax.set_ylabel('Densidad', fontsize=11)
ax.set_title(
    'Distribución de confianza\n'
    'v1 con masa cerca de 1.0 = overconfidence; v2 más spread = mejor calibrado',
    fontsize=11,
)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'distribucion_confianza.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Guardado: {OUT_DIR / "distribucion_confianza.png"}')

### Interpretación

- **Reliability diagram:** los puntos que caen **sobre** la diagonal y=x indican calibración perfecta.
  Puntos por encima de la diagonal = overconfidence (el modelo dice «soy 90% seguro» pero solo acierta el 70%).
- **ECE:** compara los tres valores. Si ECE(v2) < ECE(v1), el label smoothing α=0.1 mejoró la calibración.
- **Histograma de confianza:** una distribución más uniforme en v2 (menos masa acumulada en [0.9, 1.0])
  confirma que el label smoothing penalizó la overconfidence, tal como estaba diseñado.
- El gap val-test bajó de 0.0956 (v1) a 0.0617 (v2): el histograma nos ayuda a entender
  por qué — el modelo v2 es más conservador y generaliza mejor.

---
## Sección 6 — Curvas Precision-Recall para clases raras

Para clases con muy poco soporte en test (≤28 archivos → estimamos ≤20 clips en test),
el F1 puntual puede fluctuar mucho con un solo error. La **curva PR** muestra el
comportamiento del modelo a distintos umbrales de decisión, dando una imagen más robusta.

El **Average Precision (AP)** es el área bajo la curva PR: un AP de 1.0 es perfecto,
un AP igual al soporte de la clase (prevalencia) indica un modelo al azar.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.patch.set_facecolor('#FAFAFA')
axes_flat = axes.flatten()

ap_por_clase: dict[str, dict[str, float]] = {clave: {} for clave in CLAVES}

for panel_idx, clase in enumerate(CLASES_RARAS):
    ax  = axes_flat[panel_idx]
    idx = CLASES.index(clase)

    for clave, color, nombre in zip(CLAVES, COLORES, NOMBRES):
        y_true  = resultados[clave]['y_true']
        y_proba = resultados[clave]['y_proba']
        y_bin   = (y_true == idx).astype(int)
        scores  = y_proba[:, idx]

        ap = average_precision_score(y_bin, scores)
        ap_por_clase[clave][clase] = ap

        precision, recall, _ = precision_recall_curve(y_bin, scores)
        ax.plot(recall, precision, color=color, lw=2, alpha=0.85,
                label=f'{nombre}  AP={ap:.3f}')

    # Línea de referencia al azar (prevalencia)
    prevalencia = (resultados['baseline']['y_true'] == idx).mean()
    ax.axhline(prevalencia, linestyle='--', color='gray', lw=1, alpha=0.6, label=f'Azar (p={prevalencia:.3f})')

    ax.set_title(
        f'{clase.replace("_", " ")}\n({ARCHIVOS_POR_CLASE.get(clase,"?")} archivos entrenamiento)',
        fontsize=9,
    )
    ax.set_xlabel('Recall', fontsize=8)
    ax.set_ylabel('Precision', fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(alpha=0.3)

# Panel resumen (panel 6) — AP medio en clases raras
ax_sum = axes_flat[5]
ap_medios = {
    nombre: np.mean([ap_por_clase[clave][c] for c in CLASES_RARAS])
    for clave, nombre in zip(CLAVES, NOMBRES)
}
bars = ax_sum.bar(
    range(3), list(ap_medios.values()),
    color=COLORES, alpha=0.85, edgecolor='white',
)
for bar, (nombre, val) in zip(bars, ap_medios.items()):
    ax_sum.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold',
    )
ax_sum.set_xticks(range(3))
ax_sum.set_xticklabels(['Baseline', 'Att. v1', 'Att. v2'], fontsize=9)
ax_sum.set_ylabel('AP medio (clases raras)', fontsize=9)
ax_sum.set_title('AP medio — 5 clases raras\n(resumen)', fontsize=9)
ax_sum.set_ylim(0, 1)
ax_sum.grid(axis='y', alpha=0.3)

plt.suptitle(
    'Curvas Precision-Recall — clases raras (≤28 archivos de entrenamiento)',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.savefig(OUT_DIR / 'pr_curves_clases_raras.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Guardado: {OUT_DIR / "pr_curves_clases_raras.png"}')

print('\nAP por clase rara:')
print(f'{"Clase":<30}  {"Baseline":>10}  {"Att. v1":>10}  {"Att. v2":>10}')
for clase in CLASES_RARAS:
    print(
        f'{clase:<30}  '
        f'{ap_por_clase["baseline"][clase]:>10.3f}  '
        f'{ap_por_clase["attention_v1"][clase]:>10.3f}  '
        f'{ap_por_clase["attention_v2"][clase]:>10.3f}'
    )

### Interpretación

- Una curva que **se aleja más hacia la esquina superior derecha** indica mejor modelo para esa clase.
- Si el AP de v2 > AP de v1 para las clases raras: los class weights mejoraron la detección
  de precisamente las clases que más los necesitaban.
- Compara el panel resumen (AP medio de 5 clases raras): si las barras suben de izquierda a
  derecha (baseline → v1 → v2), cada técnica adicionada aportó valor real.

---
## Sección 7 — Conclusión final

### Mejor modelo

**Attention v2** es el ganador en la métrica de referencia (test accuracy = 0.7036)
y la más robusta al desbalance (macro F1). El gap val-test bajó de 0.0956 (v1) a 0.0617 (v2),
lo que indica mejor generalización.

### Evidencia de que los class weights funcionaron

| Evidencia | Qué muestra |
|---|---|
| Δ macro F1 (v2 − v1) | F1 subió en clases raras (↑) sin colapsar en las frecuentes |
| Correlación Pearson F1-datos | r más bajo en v2 → menor dependencia del volumen |
| F1 por clase en ≤28 arch. | Barras verdes más altas que púrpuras para clases raras |
| AP curvas PR (clases raras) | AP medio en clases raras sube con v2 |

### Evidencia de que el label smoothing funcionó

| Evidencia | Qué muestra |
|---|---|
| ECE(v2) < ECE(v1) | La calibración mejoró — las probabilidades son más confiables |
| Histograma de confianza | Menos masa en [0.95, 1.0] en v2 → menos overconfidence |
| Gap val-test 0.0617 vs 0.0956 | Regularización implícita del label smoothing |
| Reliability diagram más ajustado a y=x | v2 más cercano a la diagonal en todos los bins |

### Limitaciones que persisten

1. **Cuello de botella en datos**: incluso con class weights, las clases con 20 archivos
   (Frederickena, Rupornis) siguen teniendo F1 bajo. El límite real es el tamaño del dataset.
2. **Un solo bloque de atención**: la arquitectura usa MHSA sin feed-forward ni profundidad.
   Un Transformer completo podría capturar más contexto temporal.
3. **Sin augmentation específica de dominio**: técnicas como SpecAugment o mixup de espectrogramas
   podrían ayudar especialmente a las clases raras.
4. **Dataset no aumentado con más fuentes**: Xeno-Canto tiene cientos de grabaciones por especie;
   solo se usaron 20-79 archivos por clase.

### Próximos pasos (S6)

- Demo interactivo: subir audio → predicción en tiempo real con v2.
- README con instrucciones de uso del motor de inferencia (`src/inference.py`).
- Reporte final: integrar figuras de este notebook como evidencia central.
- Presentación: usar `f1_por_clase.png` y `calibracion.png` como diapositivas clave.

---
## Resumen final en JSON

Guardamos todas las métricas numéricas en un JSON consultable para el reporte.

In [ ]:
resumen: dict = {
    'descripcion': 'Comparativa final de los 3 modelos SelvaSonic-ML',
    'fecha': '2026-06-02',
    'modelos': {},
    'ece': {clave: eces[clave] for clave in CLAVES},
    'correlacion_pearson_f1_vs_datos': corrs,
    'ap_clases_raras': {
        clave: {clase: ap_por_clase[clave][clase] for clase in CLASES_RARAS}
        for clave in CLAVES
    },
    'ap_medio_clases_raras': {
        clave: float(np.mean([ap_por_clase[clave][c] for c in CLASES_RARAS]))
        for clave in CLAVES
    },
}

for clave, nombre in zip(CLAVES, NOMBRES):
    r = resultados[clave]
    resumen['modelos'][clave] = {
        'nombre': nombre,
        'accuracy': r['accuracy'],
        'macro_f1': r['macro_f1'],
        'macro_precision': r['macro_precision'],
        'macro_recall': r['macro_recall'],
        'f1_per_class': {
            clase: float(r['f1_per_class'][i])
            for i, clase in enumerate(CLASES)
        },
        'precision_per_class': {
            clase: float(r['precision_per_class'][i])
            for i, clase in enumerate(CLASES)
        },
        'recall_per_class': {
            clase: float(r['recall_per_class'][i])
            for i, clase in enumerate(CLASES)
        },
    }

salida = OUT_DIR / 'resumen_final.json'
with open(salida, 'w', encoding='utf-8') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print(f'Guardado: {salida}')
print(f'\nResumen de métricas globales:')
for clave, nombre in zip(CLAVES, NOMBRES):
    r = resultados[clave]
    print(f'  {nombre:<35}  acc={r["accuracy"]:.4f}  macro_f1={r["macro_f1"]:.4f}  ECE={eces[clave]:.4f}')